# Chat.DT — cloud experiment runner (Gemini / OpenAI)

Runs Settings 1–3 (`DIRECT_QA_BASELINE`, `DIRECT_QA_GROUNDED`, `CYPHER_SOFT`)
from any Colab runtime — CPU is enough — against a remote Chat.DT API.

`CYPHER_STRICT` is omitted: Outlines-constrained decoding requires a local HF
model on a GPU (use `run_experiment_local.ipynb` for that).

**Architecture**

- **Server (`api` container)** owns Neo4j (Bolt is internal-only), the test set,
  pre-executed gold queries, Cypher execution, and SVR/SCR/EA scoring.
- **This notebook** owns the LLM. It loads the server-built `bundle_<model>.json`
  to rehydrate vocabulary + IDS schema + model dump, calls the cloud LLM, and
  POSTs each generated output to `/evaluate`.

**Inputs**

1. A server-built `bundle_<model>.json` (from `scripts/build_bundle.py`).
2. The HTTPS `API_BASE_URL` of your Chat.DT deployment + the `API_BEARER_TOKEN`.
3. `LLM_API_KEY` for the chosen provider (Gemini or OpenAI).

## 1. Clone the repo and install dependencies

In [ ]:
%%bash
set -e
if [ ! -d Chat.DT ]; then
  git clone --depth 1 https://github.com/IvansSmirnoff/Chat.DT.git
fi
cd Chat.DT
pip install -q -r requirements-base.txt
# Cloud runs only need the Gemini/OpenAI SDKs — skip torch/outlines.
pip install -q google-generativeai openai

In [ ]:
import os, sys
sys.path.insert(0, 'Chat.DT')

# --- Remote Chat.DT API ---
os.environ['API_BASE_URL']     = 'https://YOUR_TUNNEL_HOST'
os.environ['API_BEARER_TOKEN'] = 'REPLACE_ME'

# --- LLM provider (switch to 'openai' if preferred) ---
os.environ['LLM_PROVIDER']    = 'gemini'
os.environ['LLM_MODEL_NAME']  = 'gemini-1.5-flash'
os.environ['LLM_API_KEY']     = 'REPLACE_ME'

## 2. Upload the bundle and ping the API

Drag `bundle_<model>.json` into the file pane. The readiness check below
catches misconfigured tokens / unreachable Neo4j before any LLM call.

In [ ]:
from pathlib import Path
from src.client.api_client import ApiClient

BUNDLE_PATH = Path('bundle_barcelona.json')   # change to your bundle filename
assert BUNDLE_PATH.exists(), f'Bundle not found: {BUNDLE_PATH}. Upload it via the file pane.'

client = ApiClient(os.environ['API_BASE_URL'], os.environ['API_BEARER_TOKEN'])
print('health:', client.health())
print('ready: ', client.health_ready())
print('test cases on server:', len(client.get_test_set()))

## 3. Run Settings 1–3 via the API

`CYPHER_STRICT` is excluded — Gemini/OpenAI cannot do Outlines regex decoding.
For the strict setting, use the GPU notebook instead.

In [ ]:
from src.config import ExperimentSetting
from src.client.runner import ApiExperimentRunner, ApiRunnerConfig

config = ApiRunnerConfig(
    bundle_path=BUNDLE_PATH,
    output_dir=Path('results'),
    name='cloud_run',
    settings=[
        ExperimentSetting.DIRECT_QA_BASELINE,
        ExperimentSetting.DIRECT_QA_GROUNDED,
        ExperimentSetting.CYPHER_SOFT,
    ],
)

runner = ApiExperimentRunner(client=client, config=config)
runner.setup()
all_rows = runner.run_comparison()

## 4. Inspect results

CSV + JSON summaries land in `results/`. Copy to Drive or download with the file browser.

In [ ]:
import json, glob
for path in sorted(glob.glob('results/*_summary.json')):
    print(path)
    print(json.dumps(json.loads(open(path).read())['metrics'], indent=2))
    print()